# DP-SGD Audit Framework Validation (Colab GPU)

This notebook is a self-contained Google Colab notebook for validating the DP upper-bound vs auditing-lower-bound framework. The code is written directly in notebook cells: no GitHub clone, no payload blob, and no hidden subprocess source strings.

For final-paper evidence, MNIST is the required dataset. CIFAR-10 and Adult are optional robustness checks; if an external dataset host is unavailable, the notebook records a warning rather than invalidating the MNIST result.


## Step 1: Prepare Lightweight Python Dependencies

This cell keeps Colab's preinstalled `torch` build, reports the currently installed versions, and only installs the small notebook-specific packages if they are missing or not at the expected version. It should run once and continue without forcing a runtime restart.


In [ ]:
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version

EXPECTED = {
    'opacus': '1.4.1',
    'dp-accounting': '0.6.0',
}
OPTIONAL = ['torch', 'numpy', 'pandas', 'scikit-learn']


def installed_version(package_name: str) -> str | None:
    try:
        return version(package_name)
    except PackageNotFoundError:
        return None


observed = {package_name: installed_version(package_name) for package_name in [*OPTIONAL, *EXPECTED.keys()]}
print('observed_versions =', observed)

missing_or_mismatched = [
    f'{package_name}=={expected_version}'
    for package_name, expected_version in EXPECTED.items()
    if observed.get(package_name) != expected_version
]

if missing_or_mismatched:
    command = [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '--upgrade',
        '--no-deps',
        *missing_or_mismatched,
    ]
    print('+', ' '.join(command))
    subprocess.run(command, check=True)
    observed = {package_name: installed_version(package_name) for package_name in [*OPTIONAL, *EXPECTED.keys()]}
    print('updated_versions =', observed)
else:
    print('Required notebook packages already available. Continuing without reinstalling torch/numpy.')


## Step 2: Imports, Paths, And Core Types

This cell imports the full stack, checks that CUDA is visible, defines the output paths, and sets up the core dataclasses used throughout the experiment.


In [ ]:
from __future__ import annotations

import csv
import gzip
import json
import math
import pickle
import random
import statistics
import tarfile
import time
import traceback
import urllib.error
import urllib.request
import warnings
import zipfile
from collections import Counter
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Mapping, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dp_accounting.pld import privacy_loss_distribution as pld_lib
from opacus import PrivacyEngine
from sklearn.datasets import fetch_openml, load_digits
from torch.utils.data import ConcatDataset, DataLoader, Subset, TensorDataset, random_split
from tqdm.auto import tqdm

OUTPUT_ROOT = Path('/content/dp_sgd_audit_framework_validation')
DATA_DIR = OUTPUT_ROOT / 'data'
MODELS_DIR = OUTPUT_ROOT / 'models'
TRAINING_DIR = OUTPUT_ROOT / 'training'
for path in [OUTPUT_ROOT, DATA_DIR, MODELS_DIR, TRAINING_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SUMMARY_JSON = OUTPUT_ROOT / 'framework_validation_gpu_scale_summary.json'
SUMMARY_CSV = OUTPUT_ROOT / 'framework_validation_gpu_scale_rows.csv'
CHECKS_JSON = OUTPUT_ROOT / 'framework_validation_gpu_scale_checks.json'
REPORT_MD = OUTPUT_ROOT / 'framework_validation_gpu_scale_report.md'
ARTIFACTS_ZIP = OUTPUT_ROOT / 'framework_validation_gpu_scale_artifacts.zip'

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this notebook.')

print('torch_version =', torch.__version__)
print('cuda_available =', torch.cuda.is_available())
print('device_count =', torch.cuda.device_count())
print('device_name =', torch.cuda.get_device_name(0))
print('arch_list =', torch.cuda.get_arch_list())
print('output_root =', OUTPUT_ROOT)


VALUE_DISCRETIZATION_INTERVAL = 1e-4

@dataclass(slots=True)
class DatasetBundle:
    train_dataset: object
    eval_dataset: object
    input_dim: int
    num_classes: int
    train_size: int
    eval_size: int


@dataclass(slots=True)
class TrainingRecord:
    training_run_id: str
    dataset: str
    model_name: str
    split_seed: int
    training_seed: int
    batch_size: int
    eval_batch_size: int
    epochs: int
    clipping_norm: float
    noise_multiplier: float
    learning_rate: float
    momentum: float
    sampling_rate: float
    delta: float
    epsilon_upper_theory: float
    epsilon_upper_pld: float | None
    pld_accounting_backend: str | None
    utility_metrics: dict[str, float]
    model_artifact_path: str | None


@dataclass(slots=True)
class TrainingOutcome:
    record: TrainingRecord
    checkpoint_path: Path | None
    model_state_dict: dict[str, Any] | None = None


@dataclass(slots=True)
class EmpiricalEpsilonEstimate:
    epsilon_lower_empirical: float
    epsilon_lower_empirical_point_estimate: float
    epsilon_lower_empirical_conservative: float
    empirical_ci_lower: float | None
    empirical_ci_upper: float | None
    estimation_method: str
    delta: float
    selected_threshold: float | None = None
    selected_event: str | None = None
    selected_event_direction: str | None = None
    selected_tpr: float | None = None
    selected_fpr: float | None = None
    member_event_fraction: float | None = None
    nonmember_event_fraction: float | None = None
    member_favoring: bool | None = None
    no_member_favoring_event_found: bool = False
    selected_event_is_tiny_tail: bool | None = None
    selected_member_event_count: int | None = None
    selected_nonmember_event_count: int | None = None
    warning_message: str | None = None
    num_member_samples: int = 0
    num_nonmember_samples: int = 0


## Step 3: Utility Functions And Privacy Accounting

These helpers handle file output, PLD accounting, and small numeric utilities used by the experiment.


In [ ]:
def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def build_run_id(prefix: str, descriptor: str, seed: int) -> str:
    cleaned = descriptor.replace(" ", "_").replace("/", "_")
    return f"{prefix}_{cleaned}_seed{seed}"


def save_json(path: str | Path, data: Any) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(data, handle, indent=2, sort_keys=True)
    return path


def write_csv(path: str | Path, rows: list[dict[str, Any]]) -> Path:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames: list[str] = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({k: "|".join(v) if isinstance(v, list) else v for k, v in row.items()})
    return path


def normalize_state_dict_keys(state_dict: Mapping[str, Any]) -> dict[str, Any]:
    normalized: dict[str, Any] = {}
    for key, value in state_dict.items():
        normalized[key.removeprefix("_module.")] = value
    return normalized


def export_inference_state_dict(model) -> dict[str, Any]:
    return normalize_state_dict_keys(model.state_dict())


def safe_float(value: Any) -> float | None:
    if value is None:
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def safe_ratio(lower: float | None, upper: float | None) -> float | None:
    if lower is None or upper is None or upper <= 0.0:
        return None
    return lower / upper


def safe_gap(upper: float | None, lower: float | None) -> float | None:
    if upper is None or lower is None:
        return None
    return upper - lower


def compute_epsilon_pld(*, noise_multiplier: float, sampling_rate: float, num_steps: int, delta: float, backend: str = "auto") -> dict[str, Any]:
    if noise_multiplier <= 0:
        raise ValueError("noise_multiplier must be positive.")
    if not (0 < sampling_rate <= 1):
        raise ValueError("sampling_rate must be in (0, 1].")
    if num_steps <= 0:
        raise ValueError("num_steps must be positive.")
    if delta <= 0:
        raise ValueError("delta must be positive.")

    if backend in ("auto", "google"):
        try:
            eps = _compute_with_dp_accounting(
                noise_multiplier=noise_multiplier,
                sampling_rate=sampling_rate,
                num_steps=num_steps,
                delta=delta,
            )
            return {"epsilon_pld": round(eps, 6), "backend_used": "google_dp_accounting", "num_steps": num_steps}
        except Exception:
            if backend == "google":
                raise

    eps = _compute_analytical_gaussian(
        noise_multiplier=noise_multiplier,
        sampling_rate=sampling_rate,
        num_steps=num_steps,
        delta=delta,
    )
    return {"epsilon_pld": round(eps, 6), "backend_used": "analytical_gaussian", "num_steps": num_steps}


def _compute_with_dp_accounting(*, noise_multiplier: float, sampling_rate: float, num_steps: int, delta: float) -> float:
    kwargs = {
        "standard_deviation": noise_multiplier,
        "pessimistic_estimate": True,
        "sampling_prob": sampling_rate,
        "use_connect_dots": True,
        "value_discretization_interval": VALUE_DISCRETIZATION_INTERVAL,
    }
    try:
        pld_single = pld_lib.from_gaussian_mechanism(sensitivity=1.0, **kwargs)
    except TypeError:
        pld_single = pld_lib.from_gaussian_mechanism(sensitivities=[1.0], **kwargs)
    pld_composed = pld_single.self_compose(num_steps)
    return float(pld_composed.get_epsilon_for_delta(delta))


def _compute_analytical_gaussian(*, noise_multiplier: float, sampling_rate: float, num_steps: int, delta: float) -> float:
    mu = sampling_rate * math.sqrt(num_steps) / noise_multiplier
    return float(_gdp_to_eps_delta(mu, delta))


def _norm_cdf(x: float) -> float:
    return 0.5 * math.erfc(-x / math.sqrt(2.0))


def _gdp_to_eps_delta(mu: float, target_delta: float) -> float:
    if mu <= 0:
        return 0.0

    def _delta_of_eps(eps: float) -> float:
        t1 = _norm_cdf(-eps / mu + mu / 2.0)
        t2 = math.exp(eps) * _norm_cdf(-eps / mu - mu / 2.0)
        return t1 - t2

    lo, hi = 0.0, mu * mu + 2.0 * mu * math.sqrt(math.log(1.0 / target_delta))
    if _delta_of_eps(0.0) <= target_delta:
        return 0.0
    for _ in range(200):
        mid = (lo + hi) / 2.0
        if _delta_of_eps(mid) > target_delta:
            lo = mid
        else:
            hi = mid
        if hi - lo < 1e-10:
            break
    return hi


## Step 4: Empirical Lower-Bound Estimation

This is the conservative empirical estimator used to turn attack scores into an auditing lower bound.


In [ ]:
def estimate_empirical_lower_bound(
    *,
    member_scores: Sequence[float] | None = None,
    nonmember_scores: Sequence[float] | None = None,
    delta: float,
    score_direction: str = "higher_is_member",
    align_event_to_score_direction: bool = True,
    require_member_favoring: bool = True,
    report_confidence_supported_lower_bound: bool = True,
    tiny_tail_fraction_threshold: float = 0.125,
    tiny_tail_min_event_count: int = 5,
) -> EmpiricalEpsilonEstimate:
    if member_scores is None or nonmember_scores is None:
        raise ValueError("Both member_scores and nonmember_scores are required.")

    member = [float(score) for score in member_scores]
    nonmember = [float(score) for score in nonmember_scores]
    if not member or not nonmember:
        raise ValueError("Member and non-member score lists must both be non-empty.")

    normalized_direction = {
        "higher": "higher_is_member",
        "lower": "lower_is_member",
    }.get(score_direction, score_direction)

    if normalized_direction == "lower_is_member":
        transformed_member = [-score for score in member]
        transformed_nonmember = [-score for score in nonmember]
        selected_event_direction = "score<threshold"
        threshold_transform = lambda value: -value
    elif normalized_direction == "higher_is_member":
        transformed_member = member
        transformed_nonmember = nonmember
        selected_event_direction = "score>=threshold"
        threshold_transform = lambda value: value
    else:
        raise ValueError(f"Unsupported score_direction: {score_direction}")

    thresholds = sorted(set(transformed_member + transformed_nonmember))
    if not thresholds:
        raise ValueError("At least one threshold is required.")

    best_point = -math.inf
    best_lower = 0.0
    best_upper = 0.0
    best_threshold = None
    best_tpr = 0.0
    best_fpr = 0.0
    best_member_favoring = False
    best_member_count = 0
    best_nonmember_count = 0

    for threshold in thresholds:
        member_successes = sum(score >= threshold for score in transformed_member)
        nonmember_successes = sum(score >= threshold for score in transformed_nonmember)
        candidate = _evaluate_member_aligned_threshold(
            member_successes=member_successes,
            member_trials=len(transformed_member),
            nonmember_successes=nonmember_successes,
            nonmember_trials=len(transformed_nonmember),
            delta=delta,
        )
        if require_member_favoring and not candidate["member_favoring"]:
            continue
        if candidate["point"] > best_point:
            best_point = candidate["point"]
            best_lower = candidate["lower"]
            best_upper = candidate["upper"]
            best_threshold = threshold
            best_tpr = candidate["tpr"]
            best_fpr = candidate["fpr"]
            best_member_favoring = bool(candidate["member_favoring"])
            best_member_count = member_successes
            best_nonmember_count = nonmember_successes

    if best_threshold is None:
        warning_message = (
            "No member-favoring event found on the scanned threshold grid; "
            "returning a conservative zero empirical lower bound."
        )
        return EmpiricalEpsilonEstimate(
            epsilon_lower_empirical=0.0,
            epsilon_lower_empirical_point_estimate=0.0,
            epsilon_lower_empirical_conservative=0.0,
            empirical_ci_lower=0.0,
            empirical_ci_upper=0.0,
            estimation_method="threshold_sweep_member_aligned_no_member_favoring_event",
            delta=delta,
            selected_event_direction=selected_event_direction,
            member_favoring=False,
            no_member_favoring_event_found=True,
            warning_message=warning_message,
            num_member_samples=len(member),
            num_nonmember_samples=len(nonmember),
        )

    point_estimate = max(0.0, best_point)
    conservative_estimate = max(0.0, best_lower)
    selected_event_is_tiny_tail = _is_tiny_tail_event(
        member_event_fraction=best_tpr,
        nonmember_event_fraction=best_fpr,
        member_event_count=best_member_count,
        nonmember_event_count=best_nonmember_count,
        member_trials=len(member),
        nonmember_trials=len(nonmember),
        fraction_threshold=tiny_tail_fraction_threshold,
        min_event_count=tiny_tail_min_event_count,
    )
    warning_message = _compose_warning_message(
        report_confidence_supported_lower_bound=report_confidence_supported_lower_bound,
        point_estimate=point_estimate,
        conservative_estimate=conservative_estimate,
        selected_event_is_tiny_tail=selected_event_is_tiny_tail,
        existing_warning=None,
    )

    return EmpiricalEpsilonEstimate(
        epsilon_lower_empirical=conservative_estimate if report_confidence_supported_lower_bound else point_estimate,
        epsilon_lower_empirical_point_estimate=point_estimate,
        epsilon_lower_empirical_conservative=conservative_estimate,
        empirical_ci_lower=max(0.0, best_lower),
        empirical_ci_upper=max(0.0, best_upper),
        estimation_method="threshold_sweep_member_aligned",
        delta=delta,
        selected_threshold=threshold_transform(best_threshold),
        selected_event=selected_event_direction,
        selected_event_direction=selected_event_direction,
        selected_tpr=best_tpr,
        selected_fpr=best_fpr,
        member_event_fraction=best_tpr,
        nonmember_event_fraction=best_fpr,
        member_favoring=best_member_favoring,
        no_member_favoring_event_found=False,
        selected_event_is_tiny_tail=selected_event_is_tiny_tail,
        selected_member_event_count=best_member_count,
        selected_nonmember_event_count=best_nonmember_count,
        warning_message=warning_message,
        num_member_samples=len(member),
        num_nonmember_samples=len(nonmember),
    )


def _evaluate_member_aligned_threshold(*, member_successes: int, member_trials: int, nonmember_successes: int, nonmember_trials: int, delta: float) -> dict[str, float | str]:
    tpr = member_successes / max(1, member_trials)
    fpr = nonmember_successes / max(1, nonmember_trials)
    tpr_lower, tpr_upper = _wilson_interval(member_successes, member_trials)
    fpr_lower, fpr_upper = _wilson_interval(nonmember_successes, nonmember_trials)

    event_positive_point = _epsilon_candidate(
        numerator_rate=tpr,
        denominator_rate=fpr,
        delta=delta,
        denominator_floor=0.5 / max(1, nonmember_trials),
    )
    event_positive_lower = _epsilon_candidate(
        numerator_rate=tpr_lower,
        denominator_rate=fpr_upper,
        delta=delta,
        denominator_floor=0.5 / max(1, nonmember_trials),
    )
    event_positive_upper = _epsilon_candidate(
        numerator_rate=tpr_upper,
        denominator_rate=fpr_lower,
        delta=delta,
        denominator_floor=0.5 / max(1, nonmember_trials),
    )
    return {
        "point": event_positive_point,
        "lower": event_positive_lower,
        "upper": event_positive_upper,
        "tpr": tpr,
        "fpr": fpr,
        "member_favoring": tpr > fpr,
    }


def _epsilon_candidate(*, numerator_rate: float, denominator_rate: float, delta: float, denominator_floor: float) -> float:
    numerator = numerator_rate - delta
    if numerator <= 0.0:
        return 0.0
    denominator = max(denominator_rate, denominator_floor)
    if denominator <= 0.0:
        return 0.0
    return max(0.0, math.log(numerator / denominator))


def _wilson_interval(successes: int, trials: int, z: float = 1.96) -> tuple[float, float]:
    if trials <= 0:
        return 0.0, 0.0
    phat = successes / trials
    denom = 1.0 + (z**2 / trials)
    center = (phat + (z**2 / (2.0 * trials))) / denom
    margin = z * math.sqrt((phat * (1.0 - phat) / trials) + (z**2 / (4.0 * trials**2))) / denom
    return max(0.0, center - margin), min(1.0, center + margin)


def _is_tiny_tail_event(*, member_event_fraction: float, nonmember_event_fraction: float, member_event_count: int, nonmember_event_count: int, member_trials: int, nonmember_trials: int, fraction_threshold: float, min_event_count: int) -> bool:
    max_fraction = max(member_event_fraction, nonmember_event_fraction)
    return (
        max_fraction <= fraction_threshold
        or member_event_count < min_event_count
        or nonmember_event_count < min_event_count
        or member_event_count == member_trials
        or nonmember_event_count == nonmember_trials
    )


def _compose_warning_message(*, report_confidence_supported_lower_bound: bool, point_estimate: float, conservative_estimate: float, selected_event_is_tiny_tail: bool, existing_warning: str | None) -> str | None:
    warnings_out: list[str] = []
    if existing_warning:
        warnings_out.append(existing_warning)
    if selected_event_is_tiny_tail:
        warnings_out.append("Selected threshold lies in a sparse tail region.")
    if report_confidence_supported_lower_bound and conservative_estimate < point_estimate:
        warnings_out.append("Reported empirical lower bound uses the confidence-supported lower side rather than the optimistic point estimate.")
    if not warnings_out:
        return None
    return " ".join(warnings_out)


## Step 5: Dataset Preparation

These helpers download and prepare MNIST, CIFAR-10, Adult, and the built-in scikit-learn Digits dataset, then package them into a common `DatasetBundle`.


In [ ]:
MNIST_BASE_URLS = [
    'https://storage.googleapis.com/cvdf-datasets/mnist/',
    'https://ossci-datasets.s3.amazonaws.com/mnist/',
    'http://yann.lecun.com/exdb/mnist/',
]
CIFAR10_URLS = [
    'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz',
]


def _as_url_list(urls: str | Sequence[str]) -> list[str]:
    if isinstance(urls, str):
        return [urls]
    return [str(url) for url in urls]


def download_file(urls: str | Sequence[str], destination: Path, retries: int = 4, timeout: int = 90) -> Path:
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    tmp_path = destination.with_suffix(destination.suffix + '.tmp')
    last_error: Exception | None = None
    for url in _as_url_list(urls):
        for attempt in range(1, retries + 1):
            try:
                print(f'Downloading {url} -> {destination} (attempt {attempt}/{retries})')
                request = urllib.request.Request(
                    url,
                    headers={'User-Agent': 'Mozilla/5.0 (Colab DP-SGD audit validation)'},
                )
                with urllib.request.urlopen(request, timeout=timeout) as response, tmp_path.open('wb') as handle:
                    while True:
                        chunk = response.read(1024 * 1024)
                        if not chunk:
                            break
                        handle.write(chunk)
                tmp_path.replace(destination)
                return destination
            except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, OSError) as exc:
                last_error = exc
                if tmp_path.exists():
                    tmp_path.unlink()
                wait_seconds = min(20, 2 ** attempt)
                print(f'  download failed: {exc}; retrying in {wait_seconds}s')
                time.sleep(wait_seconds)
    raise RuntimeError(f'Failed to download {destination.name} from all configured URLs') from last_error


def mnist_urls(filename: str) -> list[str]:
    return [base + filename for base in MNIST_BASE_URLS]


def _read_idx_images_gz(path: Path) -> np.ndarray:
    with gzip.open(path, 'rb') as handle:
        raw = handle.read()
    magic = int.from_bytes(raw[0:4], 'big')
    if magic != 2051:
        raise ValueError(f'Unexpected MNIST image magic number: {magic}')
    count = int.from_bytes(raw[4:8], 'big')
    rows = int.from_bytes(raw[8:12], 'big')
    cols = int.from_bytes(raw[12:16], 'big')
    images = np.frombuffer(raw, dtype=np.uint8, offset=16)
    return images.reshape(count, rows, cols)


def _read_idx_labels_gz(path: Path) -> np.ndarray:
    with gzip.open(path, 'rb') as handle:
        raw = handle.read()
    magic = int.from_bytes(raw[0:4], 'big')
    if magic != 2049:
        raise ValueError(f'Unexpected MNIST label magic number: {magic}')
    count = int.from_bytes(raw[4:8], 'big')
    labels = np.frombuffer(raw, dtype=np.uint8, offset=8)
    return labels.reshape(count)


def ensure_mnist_npz(target_path: Path) -> tuple[int, int]:
    if target_path.exists():
        cached = np.load(target_path)
        return int(cached['images'].shape[0]), int(cached['labels'].max()) + 1

    raw_dir = target_path.parent / 'mnist_raw'
    images_gz = download_file(mnist_urls('train-images-idx3-ubyte.gz'), raw_dir / 'train-images-idx3-ubyte.gz')
    labels_gz = download_file(mnist_urls('train-labels-idx1-ubyte.gz'), raw_dir / 'train-labels-idx1-ubyte.gz')
    images = _read_idx_images_gz(images_gz).astype(np.float32) / 255.0
    labels = _read_idx_labels_gz(labels_gz).astype(np.int64)
    images = images[:, None, :, :]
    images = (images - 0.1307) / 0.3081
    np.savez_compressed(target_path, images=images, labels=labels)
    return int(images.shape[0]), int(labels.max()) + 1


def ensure_cifar10_npz(target_path: Path) -> tuple[int, int]:
    if target_path.exists():
        cached = np.load(target_path)
        return int(cached['images'].shape[0]), int(cached['labels'].max()) + 1

    archive_path = target_path.parent / 'cifar-10-python.tar.gz'
    download_file(CIFAR10_URLS, archive_path)
    batch_arrays = []
    batch_labels = []
    with tarfile.open(archive_path, 'r:gz') as archive:
        for batch_index in range(1, 6):
            member = archive.getmember(f'cifar-10-batches-py/data_batch_{batch_index}')
            with archive.extractfile(member) as handle:
                batch = pickle.load(handle, encoding='bytes')
            batch_arrays.append(batch[b'data'])
            batch_labels.extend(batch[b'labels'])
    images = np.concatenate(batch_arrays, axis=0).reshape(-1, 3, 32, 32).astype(np.float32) / 255.0
    labels = np.asarray(batch_labels, dtype=np.int64)
    means = np.asarray([0.4914, 0.4822, 0.4465], dtype=np.float32).reshape(1, 3, 1, 1)
    stds = np.asarray([0.2470, 0.2435, 0.2616], dtype=np.float32).reshape(1, 3, 1, 1)
    images = (images - means) / stds
    np.savez_compressed(target_path, images=images, labels=labels)
    return int(images.shape[0]), int(labels.max()) + 1


def ensure_sklearn_digits_npz(target_path: Path) -> tuple[int, int]:
    if target_path.exists():
        cached = np.load(target_path)
        return int(cached['images'].shape[0]), int(cached['labels'].max()) + 1
    digits = load_digits()
    images = digits.images.astype(np.float32) / 16.0
    labels = digits.target.astype(np.int64)
    images = images[:, None, :, :]
    np.savez_compressed(target_path, images=images, labels=labels)
    return int(images.shape[0]), int(labels.max()) + 1


def fetch_openml_with_retries(name: str, version: int, retries: int = 4):
    last_error: Exception | None = None
    for attempt in range(1, retries + 1):
        try:
            return fetch_openml(name, version=version, as_frame=True)
        except Exception as exc:
            last_error = exc
            wait_seconds = min(30, 2 ** attempt)
            print(f'fetch_openml({name!r}) failed: {exc}; retrying in {wait_seconds}s')
            time.sleep(wait_seconds)
    raise RuntimeError(f'Failed to fetch OpenML dataset {name!r}') from last_error


def ensure_adult_npz(target_path: Path) -> tuple[int, int, int]:
    if target_path.exists():
        cached = np.load(target_path)
        features = cached['features']
        labels = cached['labels']
        return int(features.shape[1]), int(labels.max()) + 1, int(features.shape[0])

    target_path.parent.mkdir(parents=True, exist_ok=True)
    adult = fetch_openml_with_retries('adult', version=2)
    features = adult.data.copy()
    labels = adult.target.astype(str).copy()
    features = features.replace('?', pd.NA)
    mask = ~features.isna().any(axis=1) & labels.notna()
    features = features.loc[mask]
    labels = labels.loc[mask]
    encoded = pd.get_dummies(features, dummy_na=False, drop_first=False, dtype='float32')
    label_values = sorted(labels.unique().tolist())
    label_map = {label: index for index, label in enumerate(label_values)}
    y = labels.map(label_map).astype('int64').to_numpy()
    x = encoded.to_numpy(dtype='float32')
    np.savez_compressed(target_path, features=x, labels=y)
    save_json(
        target_path.with_suffix('.metadata.json'),
        {
            'rows': int(x.shape[0]),
            'input_dim': int(x.shape[1]),
            'num_classes': int(len(label_values)),
            'label_map': label_map,
        },
    )
    return int(x.shape[1]), int(len(label_values)), int(x.shape[0])


def load_dataset_bundle(dataset_name: str, data_dir: Path, validation_fraction: float, split_seed: int) -> DatasetBundle:
    name = dataset_name.lower()
    generator = torch.Generator().manual_seed(split_seed)
    if name == 'mnist':
        mnist_path = data_dir / 'mnist_train.npz'
        ensure_mnist_npz(mnist_path)
        data = np.load(mnist_path)
        images = torch.tensor(data['images'], dtype=torch.float32)
        labels = torch.tensor(data['labels'], dtype=torch.long)
        full_dataset = TensorDataset(images, labels)
        eval_size = int(len(full_dataset) * validation_fraction)
        train_size = len(full_dataset) - eval_size
        train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size], generator=generator)
        return DatasetBundle(train_dataset, eval_dataset, 28 * 28, 10, train_size, eval_size)
    if name == 'cifar10':
        cifar_path = data_dir / 'cifar10_train.npz'
        ensure_cifar10_npz(cifar_path)
        data = np.load(cifar_path)
        images = torch.tensor(data['images'], dtype=torch.float32)
        labels = torch.tensor(data['labels'], dtype=torch.long)
        full_dataset = TensorDataset(images, labels)
        eval_size = int(len(full_dataset) * validation_fraction)
        train_size = len(full_dataset) - eval_size
        train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size], generator=generator)
        return DatasetBundle(train_dataset, eval_dataset, 3 * 32 * 32, 10, train_size, eval_size)
    if name == 'sklearn_digits':
        digits_path = data_dir / 'sklearn_digits.npz'
        ensure_sklearn_digits_npz(digits_path)
        data = np.load(digits_path)
        images = torch.tensor(data['images'], dtype=torch.float32)
        labels = torch.tensor(data['labels'], dtype=torch.long)
        full_dataset = TensorDataset(images, labels)
        eval_size = int(len(full_dataset) * validation_fraction)
        train_size = len(full_dataset) - eval_size
        train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size], generator=generator)
        return DatasetBundle(train_dataset, eval_dataset, 8 * 8, 10, train_size, eval_size)
    if name == 'adult':
        adult_path = data_dir / 'adult.npz'
        ensure_adult_npz(adult_path)
        data = np.load(adult_path)
        features = torch.tensor(data['features'], dtype=torch.float32)
        labels = torch.tensor(data['labels'], dtype=torch.long)
        full_dataset = TensorDataset(features, labels)
        eval_size = int(len(full_dataset) * validation_fraction)
        train_size = len(full_dataset) - eval_size
        train_dataset, eval_dataset = random_split(full_dataset, [train_size, eval_size], generator=generator)
        return DatasetBundle(train_dataset, eval_dataset, int(features.shape[1]), int(labels.max().item()) + 1, train_size, eval_size)
    raise NotImplementedError(f'Unsupported dataset: {dataset_name}')


def subset_bundle(bundle: DatasetBundle, train_limit: int, eval_limit: int, seed: int) -> DatasetBundle:
    generator = torch.Generator().manual_seed(seed)
    train_perm = torch.randperm(len(bundle.train_dataset), generator=generator).tolist()
    eval_perm = torch.randperm(len(bundle.eval_dataset), generator=generator).tolist()
    train_indices = train_perm[: min(train_limit, len(train_perm))]
    eval_indices = eval_perm[: min(eval_limit, len(eval_perm))]
    return DatasetBundle(
        Subset(bundle.train_dataset, train_indices),
        Subset(bundle.eval_dataset, eval_indices),
        bundle.input_dim,
        bundle.num_classes,
        len(train_indices),
        len(eval_indices),
    )


## Step 6: Models And DP-SGD Training

This cell defines the models, evaluation helpers, and the DP-SGD training loop that produces the theoretical upper bounds.


In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int) -> None:
        super().__init__()
        self.network = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, inputs):
        return self.network(inputs)


class SimpleCNN(nn.Module):
    def __init__(self, num_classes: int = 10) -> None:
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.pool1(x)
        x = self.relu(self.conv2(x))
        x = self.pool2(x)
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x


class TabularMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_classes: int) -> None:
        super().__init__()
        mid_dim = max(16, hidden_dim // 2)
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, mid_dim),
            nn.ReLU(),
            nn.Linear(mid_dim, num_classes),
        )

    def forward(self, x):
        return self.network(x)


def build_model(model_name: str, input_dim: int, hidden_dim: int, num_classes: int):
    name = model_name.lower()
    if name == "simple_mlp":
        return SimpleMLP(input_dim, hidden_dim, num_classes)
    if name == "cnn_cifar10":
        return SimpleCNN(num_classes=num_classes)
    if name == "tabular_mlp":
        return TabularMLP(input_dim=input_dim, hidden_dim=hidden_dim, num_classes=num_classes)
    raise NotImplementedError(f"Unsupported model: {model_name}")


def load_model_from_state_dict(spec: dict[str, Any], state_dict: Mapping[str, Any], device=None):
    resolved_device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = build_model(spec["model_name"], spec["input_dim"], spec["hidden_dim"], spec["num_classes"]).to(resolved_device)
    model.load_state_dict(normalize_state_dict_keys(state_dict))
    model.eval()
    return model


def evaluate_classifier(model, data_loader, device) -> dict[str, float]:
    model.eval()
    correct = 0
    total = 0
    cumulative_loss = 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            logits = model(inputs)
            loss = criterion(logits, targets)
            cumulative_loss += float(loss.item()) * targets.size(0)
            predictions = logits.argmax(dim=1)
            correct += int((predictions == targets).sum().item())
            total += int(targets.size(0))
    return {"accuracy": correct / max(1, total), "loss": cumulative_loss / max(1, total)}


def train_dp_sgd(spec: dict[str, Any], bundle: DatasetBundle, *, save_checkpoint: bool = True, run_descriptor: str | None = None, return_model_state: bool = False) -> TrainingOutcome:
    set_global_seed(spec["training_seed"])
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pin_memory = device.type == "cuda"
    train_loader = DataLoader(bundle.train_dataset, batch_size=spec["batch_size"], shuffle=True, num_workers=2, pin_memory=pin_memory)
    eval_loader = DataLoader(bundle.eval_dataset, batch_size=spec["eval_batch_size"], shuffle=False, num_workers=2, pin_memory=pin_memory)
    model = build_model(spec["model_name"], spec["input_dim"], spec["hidden_dim"], spec["num_classes"]).to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=spec["learning_rate"], momentum=spec["momentum"], weight_decay=0.0)
    criterion = nn.CrossEntropyLoss()
    privacy_engine = PrivacyEngine(accountant="rdp")
    model, optimizer, train_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=train_loader,
        noise_multiplier=spec["noise_multiplier"],
        max_grad_norm=spec["clipping_norm"],
    )
    for epoch_index in range(spec["epochs"]):
        model.train()
        running_loss = 0.0
        for inputs, targets in train_loader:
            inputs = inputs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            logits = model(inputs)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()
            running_loss += float(loss.item())
        print(f"[{spec['dataset_name']}] epoch {epoch_index + 1}/{spec['epochs']} avg_loss={running_loss / max(1, len(train_loader)):.4f}")
    utility_metrics = evaluate_classifier(model=model, data_loader=eval_loader, device=device)
    sampling_rate = spec["batch_size"] / bundle.train_size
    epsilon_upper_theory = float(privacy_engine.accountant.get_epsilon(spec["delta"]))
    num_steps = spec["epochs"] * len(train_loader)
    pld_result = compute_epsilon_pld(
        noise_multiplier=spec["noise_multiplier"],
        sampling_rate=sampling_rate,
        num_steps=num_steps,
        delta=spec["delta"],
    )
    run_id = build_run_id("train", run_descriptor or spec["experiment_name"], spec["training_seed"])
    inference_state_dict = export_inference_state_dict(model)
    checkpoint_path = None
    if save_checkpoint:
        checkpoint_path = MODELS_DIR / f"{run_id}.pt"
        torch.save(inference_state_dict, checkpoint_path)
    record = TrainingRecord(
        training_run_id=run_id,
        dataset=spec["dataset_name"],
        model_name=spec["model_name"],
        split_seed=spec["split_seed"],
        training_seed=spec["training_seed"],
        batch_size=spec["batch_size"],
        eval_batch_size=spec["eval_batch_size"],
        epochs=spec["epochs"],
        clipping_norm=spec["clipping_norm"],
        noise_multiplier=spec["noise_multiplier"],
        learning_rate=spec["learning_rate"],
        momentum=spec["momentum"],
        sampling_rate=sampling_rate,
        delta=spec["delta"],
        epsilon_upper_theory=epsilon_upper_theory,
        epsilon_upper_pld=pld_result["epsilon_pld"],
        pld_accounting_backend=pld_result["backend_used"],
        utility_metrics=utility_metrics,
        model_artifact_path=str(checkpoint_path) if checkpoint_path else None,
    )
    save_json(TRAINING_DIR / f"{run_id}.json", asdict(record))
    return TrainingOutcome(
        record=record,
        checkpoint_path=checkpoint_path,
        model_state_dict=inference_state_dict if return_model_state else None,
    )


## Step 7: Passive Auditing Helpers

This section defines the passive negative-loss baseline and the Raw LiRA shadow-model machinery.


In [ ]:
def sample_query_indices(train_size: int, eval_size: int, audit_seeds: list[int], query_budget: int):
    all_train_indices: set[int] = set()
    all_eval_indices: set[int] = set()
    per_seed_train: dict[int, list[int]] = {}
    per_seed_eval: dict[int, list[int]] = {}
    half_budget = min(query_budget // 2, train_size, eval_size)
    if half_budget < query_budget // 2:
        print(f'Using reduced per-class query budget {half_budget} because dataset split is small.')
    for seed in audit_seeds:
        rng = random.Random(seed)
        train_indices = rng.sample(range(train_size), half_budget)
        eval_indices = rng.sample(range(eval_size), half_budget)
        per_seed_train[seed] = train_indices
        per_seed_eval[seed] = eval_indices
        all_train_indices.update(train_indices)
        all_eval_indices.update(eval_indices)
    return sorted(all_train_indices), sorted(all_eval_indices), per_seed_train, per_seed_eval


def compute_loss_for_indices(model, dataset, indices, device, batch_size=256):
    if not indices:
        return []
    losses = []
    with torch.no_grad():
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start : start + batch_size]
            images = []
            labels = []
            for index in batch_indices:
                image, label = dataset[index]
                images.append(image)
                labels.append(label)
            x = torch.stack(images).to(device)
            y = torch.tensor(labels, dtype=torch.long, device=device)
            batch_losses = F.cross_entropy(model(x), y, reduction="none")
            losses.extend(float(value) for value in batch_losses.detach().cpu().tolist())
    return losses


def negative_loss_scores(*, query_train_indices, query_eval_indices, per_seed_train, per_seed_eval, target_train_losses, target_eval_losses, audit_seeds):
    train_pos = {index: pos for pos, index in enumerate(query_train_indices)}
    eval_pos = {index: pos for pos, index in enumerate(query_eval_indices)}
    member_scores: list[float] = []
    nonmember_scores: list[float] = []
    for seed in audit_seeds:
        member_scores.extend(-float(target_train_losses[train_pos[index]]) for index in per_seed_train[seed])
        nonmember_scores.extend(-float(target_eval_losses[eval_pos[index]]) for index in per_seed_eval[seed])
    return member_scores, nonmember_scores


def clone_shadow_spec(spec: dict[str, Any], training_seed: int, shadow_index: int) -> dict[str, Any]:
    shadow_spec = dict(spec)
    shadow_spec["training_seed"] = training_seed
    shadow_spec["experiment_name"] = f"{spec['experiment_name']}_shadow_{shadow_index}"
    return shadow_spec


def train_shadow_losses(*, spec: dict[str, Any], bundle: DatasetBundle, dataset_name: str, query_train_indices, query_eval_indices, k_max: int, shadow_subset_fraction: float):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    shadow_train_losses: list[list[float]] = []
    shadow_eval_losses: list[list[float]] = []
    shadow_member_sets: list[set[int]] = []
    train_size = len(bundle.train_dataset)
    shadow_train_size = max(2, int(train_size * shadow_subset_fraction))
    for shadow_index in tqdm(range(k_max), desc=f"{dataset_name} shadows", leave=False):
        shadow_seed = 5000 + shadow_index
        rng = random.Random(shadow_seed)
        all_indices = list(range(train_size))
        rng.shuffle(all_indices)
        in_indices = sorted(all_indices[:shadow_train_size])
        shadow_member_sets.append(set(in_indices))
        shadow_bundle = DatasetBundle(
            train_dataset=Subset(bundle.train_dataset, in_indices),
            eval_dataset=bundle.eval_dataset,
            input_dim=bundle.input_dim,
            num_classes=bundle.num_classes,
            train_size=len(in_indices),
            eval_size=bundle.eval_size,
        )
        shadow_spec = clone_shadow_spec(spec, training_seed=shadow_seed, shadow_index=shadow_index)
        outcome = train_dp_sgd(
            shadow_spec,
            shadow_bundle,
            save_checkpoint=False,
            run_descriptor=f"{dataset_name}_shadow_{shadow_index}",
            return_model_state=True,
        )
        shadow_model = load_model_from_state_dict(spec, outcome.model_state_dict, device=device)
        shadow_train_losses.append(compute_loss_for_indices(shadow_model, bundle.train_dataset, query_train_indices, device=device, batch_size=spec["eval_batch_size"]))
        shadow_eval_losses.append(compute_loss_for_indices(shadow_model, bundle.eval_dataset, query_eval_indices, device=device, batch_size=spec["eval_batch_size"]))
    return shadow_train_losses, shadow_eval_losses, shadow_member_sets


def raw_lira_scores(*, query_train_indices, query_eval_indices, per_seed_train, per_seed_eval, target_train_losses, target_eval_losses, shadow_train_losses, shadow_eval_losses, shadow_member_sets, k: int, audit_seeds: list[int]):
    train_pos = {index: pos for pos, index in enumerate(query_train_indices)}
    eval_pos = {index: pos for pos, index in enumerate(query_eval_indices)}
    member_scores: list[float] = []
    nonmember_scores: list[float] = []
    for seed in audit_seeds:
        for index in per_seed_train[seed]:
            pos = train_pos[index]
            in_losses = []
            out_losses = []
            for shadow_index in range(k):
                loss_value = float(shadow_train_losses[shadow_index][pos])
                if index in shadow_member_sets[shadow_index]:
                    in_losses.append(loss_value)
                else:
                    out_losses.append(loss_value)
            if in_losses and out_losses:
                score = statistics.fmean(out_losses) - statistics.fmean(in_losses)
            elif out_losses:
                score = statistics.fmean(out_losses)
            else:
                score = -statistics.fmean(in_losses) if in_losses else 0.0
            member_scores.append(float(score))
        for index in per_seed_eval[seed]:
            pos = eval_pos[index]
            shadow_losses = [float(shadow_eval_losses[shadow_index][pos]) for shadow_index in range(k)]
            shadow_mean = statistics.fmean(shadow_losses) if shadow_losses else 0.0
            score = shadow_mean - float(target_eval_losses[pos])
            nonmember_scores.append(float(score))
    return member_scores, nonmember_scores


## Step 8: Canary Auditing Helpers

This section defines the image-canary generation and active canary audit used as an additional stress test.


In [ ]:
@dataclass(slots=True)
class CanaryExample:
    identifier: str
    target_label: int
    inserted_image: object
    reference_image: object
    descriptor: str


@dataclass(slots=True)
class CanaryInsertionResult:
    augmented_train_dataset: object
    inserted_canaries: list[CanaryExample]
    reference_canaries: list[CanaryExample]
    inserted_example_count: int
    unique_inserted_count: int


def build_canary_seed_plan(*, experiment_seed: int, audit_seed: int, dataset_split_seed: int) -> dict[str, int]:
    def derive(stream: int) -> int:
        seed = (abs(experiment_seed) * 1_000_003 + abs(audit_seed) * 9_176 + stream * 104_729 + 17) % 2_147_483_647
        return seed if seed != 0 else stream + 1
    return {
        "experiment_seed": experiment_seed,
        "audit_seed": audit_seed,
        "dataset_split_seed": dataset_split_seed,
        "canary_generation_seed": derive(11),
        "canary_insertion_seed": derive(23),
        "retrain_seed": derive(37),
    }


def _infer_image_shape(dataset_name: str) -> tuple[int, int, int]:
    if dataset_name.lower() == "mnist":
        return (1, 28, 28)
    if dataset_name.lower() == "cifar10":
        return (3, 32, 32)
    if dataset_name.lower() == "sklearn_digits":
        return (1, 8, 8)
    return (1, 28, 28)


def _build_canary_image(*, strategy: str, target_label: int, seed: int, inserted: bool, image_shape: tuple[int, ...]):
    channels, height, width = image_shape
    generator = torch.Generator().manual_seed(seed)
    image = torch.rand((channels, height, width), generator=generator) * 0.04
    patch_size = {"random_canaries": 3, "improved_canaries": 4, "optimized_canaries": 5}.get(strategy, 3)
    intensity = {"random_canaries": 0.8, "improved_canaries": 0.95, "optimized_canaries": 1.0}.get(strategy, 0.8)
    base_row = 2 + ((target_label * 3 + seed) % (height - patch_size - 2))
    base_col = 2 + ((target_label * 5 + seed // 2) % (width - patch_size - 2))
    if not inserted:
        base_row = (base_row + 7) % (height - patch_size - 1)
        base_col = (base_col + 11) % (width - patch_size - 1)
        intensity *= 0.75
    image[:, base_row : base_row + patch_size, base_col : base_col + patch_size] = intensity
    return image.clamp(0.0, 1.0), f"label={target_label},row={base_row},col={base_col},inserted={inserted}"


def generate_canaries(strategy: str, num_canaries: int, seed: int, *, num_classes: int, image_shape: tuple[int, ...]):
    rng = random.Random(seed)
    canaries = []
    for index in range(num_canaries):
        target_label = rng.randrange(num_classes)
        inserted_image, inserted_descriptor = _build_canary_image(strategy=strategy, target_label=target_label, seed=seed + index, inserted=True, image_shape=image_shape)
        reference_image, reference_descriptor = _build_canary_image(strategy=strategy, target_label=target_label, seed=seed + index + 10_000, inserted=False, image_shape=image_shape)
        canaries.append(
            CanaryExample(
                identifier=f"{strategy}_{index}",
                target_label=target_label,
                inserted_image=inserted_image.to(dtype=torch.float32),
                reference_image=reference_image.to(dtype=torch.float32),
                descriptor=f"inserted={inserted_descriptor};reference={reference_descriptor}",
            )
        )
    return canaries


def insert_canaries_into_dataset(dataset: object, canaries: list[CanaryExample], *, insertion_rate: float, seed: int | None = None) -> CanaryInsertionResult:
    inserted_example_count = max(1, int(round(len(dataset) * insertion_rate)))
    unique_inserted_count = min(len(canaries), inserted_example_count)
    selected_canaries = list(canaries)
    if seed is not None:
        random.Random(seed).shuffle(selected_canaries)
    selected_canaries = selected_canaries[:unique_inserted_count]
    inserted_images = []
    inserted_labels = []
    cycle_index = 0
    for _ in range(inserted_example_count):
        canary = selected_canaries[cycle_index % len(selected_canaries)]
        inserted_images.append(canary.inserted_image.clone())
        inserted_labels.append(canary.target_label)
        cycle_index += 1
    tensor_dataset = TensorDataset(torch.stack(inserted_images), torch.tensor(inserted_labels, dtype=torch.long))
    augmented_train_dataset = ConcatDataset([dataset, tensor_dataset])
    return CanaryInsertionResult(
        augmented_train_dataset=augmented_train_dataset,
        inserted_canaries=selected_canaries,
        reference_canaries=selected_canaries,
        inserted_example_count=inserted_example_count,
        unique_inserted_count=unique_inserted_count,
    )


def _target_margin_scores(logits, labels) -> list[float]:
    target_logits = logits.gather(1, labels.unsqueeze(1)).squeeze(1)
    mask = torch.ones_like(logits, dtype=torch.bool)
    mask.scatter_(1, labels.unsqueeze(1), False)
    other_logits = logits.masked_fill(~mask, float("-inf"))
    strongest_other = other_logits.max(dim=1).values
    return (target_logits - strongest_other).detach().cpu().tolist()


def _score_canaries(model, canaries, *, device) -> list[float]:
    with torch.no_grad():
        images = torch.stack([canary.inserted_image for canary in canaries]).to(device)
        labels = torch.tensor([canary.target_label for canary in canaries], dtype=torch.long, device=device)
        logits = model(images)
        return _target_margin_scores(logits, labels)


def _score_reference_canaries(model, canaries, *, device) -> list[float]:
    with torch.no_grad():
        images = torch.stack([canary.reference_image for canary in canaries]).to(device)
        labels = torch.tensor([canary.target_label for canary in canaries], dtype=torch.long, device=device)
        logits = model(images)
        return _target_margin_scores(logits, labels)


def run_canary_random_audit(*, spec: dict[str, Any], base_bundle: DatasetBundle, target_record: TrainingRecord, num_canaries: int, audit_seeds: list[int]) -> dict[str, Any]:
    if spec["dataset_name"] not in {"mnist", "cifar10", "sklearn_digits"}:
        return {"dataset": spec["dataset_name"], "attack": "canary_random", "support_label": "gpu_scale", "status": "not_supported", "reason": "canary validation currently supports image datasets only"}
    member_scores: list[float] = []
    nonmember_scores: list[float] = []
    epsilon_upper_rdp_values: list[float] = []
    epsilon_upper_tighter_values: list[float] = []
    inserted_counts: list[int] = []
    image_shape = _infer_image_shape(spec["dataset_name"])
    device = torch.device("cuda")
    for audit_seed in audit_seeds:
        seed_plan = build_canary_seed_plan(experiment_seed=target_record.training_seed, audit_seed=audit_seed, dataset_split_seed=target_record.split_seed)
        canaries = generate_canaries("random_canaries", num_canaries, seed_plan["canary_generation_seed"], num_classes=spec["num_classes"], image_shape=image_shape)
        insertion_result = insert_canaries_into_dataset(base_bundle.train_dataset, canaries, insertion_rate=0.005, seed=seed_plan["canary_insertion_seed"])
        augmented_bundle = DatasetBundle(
            train_dataset=insertion_result.augmented_train_dataset,
            eval_dataset=base_bundle.eval_dataset,
            input_dim=base_bundle.input_dim,
            num_classes=base_bundle.num_classes,
            train_size=base_bundle.train_size + insertion_result.inserted_example_count,
            eval_size=base_bundle.eval_size,
        )
        canary_spec = dict(spec)
        canary_spec["training_seed"] = seed_plan["retrain_seed"]
        canary_spec["experiment_name"] = f"{spec['experiment_name']}_canary_random"
        outcome = train_dp_sgd(canary_spec, augmented_bundle, save_checkpoint=False, run_descriptor=f"{spec['dataset_name']}_canary_random_{audit_seed}", return_model_state=True)
        model = load_model_from_state_dict(spec, outcome.model_state_dict, device=device)
        member_scores.extend(_score_canaries(model, insertion_result.inserted_canaries, device=device))
        nonmember_scores.extend(_score_reference_canaries(model, insertion_result.reference_canaries, device=device))
        epsilon_upper_rdp_values.append(outcome.record.epsilon_upper_theory)
        epsilon_upper_tighter_values.append(outcome.record.epsilon_upper_pld)
        inserted_counts.append(insertion_result.inserted_example_count)
    estimate = estimate_empirical_lower_bound(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        delta=spec["delta"],
        score_direction="higher_is_member",
        align_event_to_score_direction=True,
        require_member_favoring=True,
        report_confidence_supported_lower_bound=True,
    )
    epsilon_upper_rdp = statistics.fmean(epsilon_upper_rdp_values)
    epsilon_upper_tighter = statistics.fmean(epsilon_upper_tighter_values)
    epsilon_lower = estimate.epsilon_lower_empirical
    return {
        "dataset": spec["dataset_name"],
        "attack": "canary_random",
        "support_label": "gpu_scale",
        "status": "ok",
        "audit_regime": "evaluator_controlled_canary_stress_test",
        "score_name": "target_label_logit_margin",
        "num_canaries_per_seed": num_canaries,
        "num_audit_seeds": len(audit_seeds),
        "audit_seeds": json.dumps(audit_seeds),
        "mean_inserted_example_count": round(statistics.fmean(inserted_counts), 3),
        "epsilon_upper_rdp": epsilon_upper_rdp,
        "epsilon_upper_tighter": epsilon_upper_tighter,
        "tighter_upper_backend": "google_dp_accounting",
        "epsilon_lower_conservative": epsilon_lower,
        "epsilon_lower_point": estimate.epsilon_lower_empirical_point_estimate,
        "tightness_ratio_rdp": safe_ratio(epsilon_lower, epsilon_upper_rdp),
        "tightness_ratio_tighter": safe_ratio(epsilon_lower, epsilon_upper_tighter),
        "privacy_loss_gap_rdp": safe_gap(epsilon_upper_rdp, epsilon_lower),
        "privacy_loss_gap_tighter": safe_gap(epsilon_upper_tighter, epsilon_lower),
        "member_favoring": estimate.member_favoring,
        "selected_tpr": estimate.selected_tpr,
        "selected_fpr": estimate.selected_fpr,
        "warning": estimate.warning_message,
        "num_member_samples": estimate.num_member_samples,
        "num_nonmember_samples": estimate.num_nonmember_samples,
    }


## Step 9: Validation Rules And Report Generation

These helpers normalize rows, assign trust tiers, run validation checks, and write the final report plus artifact bundle.


In [ ]:
def normalize_attack_row(row: dict[str, Any]) -> dict[str, Any]:
    attack_name = str(row.get("attack"))
    status = str(row.get("status", ""))
    direction_map = {
        "passive_negative_loss": "higher_is_member",
        "passive_negative_loss_matched": "higher_is_member",
        "passive_raw_lira": "lower_is_member",
        "canary_random": "higher_is_member",
    }
    score_name_map = {
        "passive_negative_loss": "negative_loss",
        "passive_negative_loss_matched": "negative_loss",
        "passive_raw_lira": "raw_lira_score",
        "canary_random": "target_label_logit_margin",
    }
    attack_family_map = {
        "passive_negative_loss": "passive_threshold",
        "passive_negative_loss_matched": "passive_threshold",
        "passive_raw_lira": "passive_lira",
        "canary_random": "active_canary",
    }
    tags: list[str] = []
    validation_status = "executed"
    expected_limitation = False
    if status == "not_supported":
        validation_status = "expected_not_supported"
        expected_limitation = True
        tags.append("expected_capability_limit")
    elif status != "ok":
        validation_status = "failed_execution"
        tags.append("execution_failure")
    warning = str(row.get("warning") or "").lower()
    if "sparse tail" in warning:
        tags.append("sparse_tail")
    num_member = int(row.get("num_member_samples") or 0)
    num_nonmember = int(row.get("num_nonmember_samples") or 0)
    min_support = min(num_member, num_nonmember)
    if status == "ok":
        if min_support < 128:
            tags.append("low_support")
        elif min_support < 1000:
            tags.append("medium_support")
        else:
            tags.append("high_support")
    epsilon_upper_tighter = safe_float(row.get("epsilon_upper_tighter"))
    epsilon_lower_conservative = safe_float(row.get("epsilon_lower_conservative"))
    epsilon_lower_point = safe_float(row.get("epsilon_lower_point"))
    if status == "ok" and epsilon_upper_tighter is not None and epsilon_lower_conservative is not None and epsilon_lower_conservative > epsilon_upper_tighter + 0.05:
        tags.extend(["pathological_distribution", "exceeds_theoretical_upper"])
    selected_tpr = safe_float(row.get("selected_tpr"))
    selected_fpr = safe_float(row.get("selected_fpr"))
    if selected_tpr is not None and selected_tpr <= 0.02:
        tags.append("extreme_tail_threshold")
    if selected_fpr is not None and selected_fpr == 0.0:
        tags.append("zero_fpr_threshold")
    if attack_name == "passive_raw_lira":
        tags.append("score_direction_sensitive")
    if attack_name == "canary_random":
        tags.append("active_stress_test")
    if status == "ok" and epsilon_lower_conservative == 0.0 and (epsilon_lower_point or 0.0) > 0.0:
        tags.append("finite_sample_limited_signal")
    if expected_limitation:
        result_trust = "not_applicable"
    elif status != "ok":
        result_trust = "invalidated"
    elif "pathological_distribution" in tags:
        result_trust = "invalidated"
    elif "sparse_tail" in tags and ("zero_fpr_threshold" in tags or "extreme_tail_threshold" in tags):
        result_trust = "exploratory"
    elif epsilon_lower_conservative is not None and epsilon_lower_conservative > 0.0:
        result_trust = "provisional"
    else:
        result_trust = "exploratory"
    return {
        "dataset": row.get("dataset"),
        "attack_name": attack_name,
        "attack_family": attack_family_map.get(attack_name, "other"),
        "status": status,
        "validation_status": validation_status,
        "expected_limitation": expected_limitation,
        "score_name": row.get("score_name") or score_name_map.get(attack_name),
        "score_direction": direction_map.get(attack_name),
        "upper_bound_backend": row.get("tighter_upper_backend"),
        "epsilon_upper_rdp": safe_float(row.get("epsilon_upper_rdp")),
        "epsilon_upper_tighter": epsilon_upper_tighter,
        "epsilon_lower_conservative": epsilon_lower_conservative,
        "epsilon_lower_point": epsilon_lower_point,
        "selected_tpr": selected_tpr,
        "selected_fpr": selected_fpr,
        "num_member_samples": num_member,
        "num_nonmember_samples": num_nonmember,
        "query_budget_per_seed": row.get("query_budget_per_seed"),
        "num_audit_seeds": row.get("num_audit_seeds"),
        "audit_seeds": row.get("audit_seeds"),
        "k_shadows": row.get("k_shadows"),
        "num_canaries_per_seed": row.get("num_canaries_per_seed"),
        "mean_inserted_example_count": row.get("mean_inserted_example_count"),
        "warning": row.get("warning"),
        "diagnostic_tags": sorted(set(tags)),
        "result_trust": result_trust,
    }


def add_check(checks: list[dict[str, Any]], *, check_id: str, status: str, description: str, scope: str, details: str) -> None:
    checks.append({"check_id": check_id, "status": status, "description": description, "scope": scope, "details": details})


def support_profile(row: dict[str, Any]) -> str:
    pieces: list[str] = []
    if row.get("query_budget_per_seed"):
        pieces.append(f"budget={row['query_budget_per_seed']}")
    if row.get("num_audit_seeds"):
        pieces.append(f"seeds={row['num_audit_seeds']}")
    if row.get("k_shadows"):
        pieces.append(f"K={row['k_shadows']}")
    if row.get("num_canaries_per_seed"):
        pieces.append(f"canaries={row['num_canaries_per_seed']}")
    return ", ".join(pieces) or "unspecified"


REQUIRED_DATASETS = {"mnist"}
OPTIONAL_DATASETS = {"sklearn_digits", "cifar10", "adult"}
ATTACKS_BY_DATASET_TYPE = {
    "default": ["passive_negative_loss", "passive_negative_loss_matched", "passive_raw_lira"],
    "image": ["passive_negative_loss", "passive_negative_loss_matched", "passive_raw_lira", "canary_random"],
}
IMAGE_DATASETS = {"mnist", "sklearn_digits", "cifar10"}


def expected_attacks_for_dataset(dataset: str) -> list[str]:
    return ATTACKS_BY_DATASET_TYPE["image"] if dataset in IMAGE_DATASETS else ATTACKS_BY_DATASET_TYPE["default"] + ["canary_random"]


def build_checks(training_rows: list[dict[str, Any]], audit_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    checks: list[dict[str, Any]] = []
    expected_training = REQUIRED_DATASETS | OPTIONAL_DATASETS
    observed_training_ok = {row["dataset"] for row in training_rows if row.get("status") == "ok"}
    missing_required = REQUIRED_DATASETS - observed_training_ok
    missing_optional = OPTIONAL_DATASETS - observed_training_ok
    coverage_status = "fail" if missing_required else ("warn" if missing_optional else "pass")
    add_check(checks, check_id="training_matrix_coverage", status=coverage_status, description="Required final-paper datasets train successfully; optional robustness datasets are reported separately.", scope="global", details=f"observed_ok={sorted(observed_training_ok)} required={sorted(REQUIRED_DATASETS)} optional_missing={sorted(missing_optional)}")
    for row in training_rows:
        dataset = row["dataset"]
        if row.get("status") != "ok":
            status = "fail" if dataset in REQUIRED_DATASETS else "warn"
            add_check(checks, check_id=f"training_status::{dataset}", status=status, description="Training completed successfully for required datasets; optional dataset failures are warnings.", scope=dataset, details=f"error={row.get('error')}")
            continue
        backend = row.get("upper_bound_backend")
        upper_rdp = safe_float(row.get("epsilon_upper_rdp"))
        upper_tighter = safe_float(row.get("epsilon_upper_tighter"))
        add_check(checks, check_id=f"upper_backend::{dataset}", status="pass" if backend == "google_dp_accounting" else "fail", description="Exact PLD backend is active.", scope=dataset, details=f"backend={backend}")
        add_check(checks, check_id=f"upper_bound_order::{dataset}", status="pass" if upper_tighter is not None and upper_rdp is not None and upper_tighter > 0.0 and upper_tighter <= upper_rdp else "fail", description="Tighter upper bound is positive and no larger than the RDP upper bound.", scope=dataset, details=f"epsilon_upper_tighter={upper_tighter}, epsilon_upper_rdp={upper_rdp}")
    rows_by_key = {(row["dataset"], row["attack_name"]): row for row in audit_rows}
    attempted_datasets = {row["dataset"] for row in training_rows} | {row["dataset"] for row in audit_rows}
    expected_rows = {(dataset, attack) for dataset in attempted_datasets for attack in expected_attacks_for_dataset(dataset)}
    add_check(checks, check_id="audit_matrix_coverage", status="pass" if set(rows_by_key) == expected_rows else "fail", description="All canonical audit rows are present.", scope="global", details=f"observed={len(rows_by_key)} expected={len(expected_rows)}")
    for (dataset, attack), row in sorted(rows_by_key.items()):
        scope = f"{dataset}::{attack}"
        if row["validation_status"] == "expected_not_supported":
            add_check(checks, check_id=f"expected_limitation::{scope}", status="pass", description="Unsupported capability is explicit rather than silent.", scope=scope, details="status=not_supported")
            continue
        if row["validation_status"] != "executed":
            status = "fail" if dataset in REQUIRED_DATASETS else "warn"
            add_check(checks, check_id=f"execution::{scope}", status=status, description="Audit row executed successfully for required datasets; optional dataset failures are warnings.", scope=scope, details=f"status={row['status']}")
            continue
        add_check(checks, check_id=f"score_direction::{scope}", status="pass" if row.get("score_direction") else "fail", description="Score direction is explicit.", scope=scope, details=f"score_direction={row.get('score_direction')}")
        add_check(checks, check_id=f"sample_counts::{scope}", status="pass" if row["num_member_samples"] > 0 and row["num_nonmember_samples"] > 0 else "fail", description="Sample counts are positive.", scope=scope, details=f"num_member_samples={row['num_member_samples']}, num_nonmember_samples={row['num_nonmember_samples']}")
        lower_cons = row.get("epsilon_lower_conservative")
        lower_point = row.get("epsilon_lower_point")
        add_check(checks, check_id=f"lower_bound_order::{scope}", status="pass" if lower_cons is not None and lower_point is not None and lower_cons <= lower_point + 1e-9 else "fail", description="Conservative lower bound does not exceed the point estimate.", scope=scope, details=f"epsilon_lower_conservative={lower_cons}, epsilon_lower_point={lower_point}")
        tpr = row.get("selected_tpr")
        fpr = row.get("selected_fpr")
        rates_ok = ((tpr is None or 0.0 <= tpr <= 1.0) and (fpr is None or 0.0 <= fpr <= 1.0))
        add_check(checks, check_id=f"rates_in_range::{scope}", status="pass" if rates_ok else "fail", description="Selected rates are valid probabilities.", scope=scope, details=f"selected_tpr={tpr}, selected_fpr={fpr}")
        upper = row.get("epsilon_upper_tighter")
        overshoot = lower_cons is not None and upper is not None and lower_cons > upper + 0.05
        if overshoot:
            add_check(checks, check_id=f"pathology_guard::{scope}", status="pass" if row["result_trust"] == "invalidated" else "fail", description="Empirical overshoot is caught and invalidated rather than accepted.", scope=scope, details=f"epsilon_lower_conservative={lower_cons}, epsilon_upper_tighter={upper}, trust={row['result_trust']}")
        else:
            add_check(checks, check_id=f"upper_vs_lower::{scope}", status="pass", description="Empirical lower bound does not materially exceed the theoretical upper bound.", scope=scope, details=f"epsilon_lower_conservative={lower_cons}, epsilon_upper_tighter={upper}")
    for dataset in sorted(attempted_datasets):
        baseline = rows_by_key.get((dataset, "passive_negative_loss"))
        matched = rows_by_key.get((dataset, "passive_negative_loss_matched"))
        if not baseline or not matched:
            status = "fail" if dataset in REQUIRED_DATASETS else "warn"
            add_check(checks, check_id=f"baseline_consistency::{dataset}", status=status, description="Matched negative-loss agrees with the canonical passive baseline.", scope=dataset, details="missing one or both rows")
            continue
        diff = abs(float(baseline["epsilon_lower_conservative"] or 0.0) - float(matched["epsilon_lower_conservative"] or 0.0))
        add_check(checks, check_id=f"baseline_consistency::{dataset}", status="pass" if diff <= 1e-9 else "warn", description="Matched negative-loss agrees with the canonical passive baseline.", scope=dataset, details=f"absolute_difference={diff}")
    for dataset in sorted(IMAGE_DATASETS & attempted_datasets):
        baseline = rows_by_key.get((dataset, "passive_negative_loss"))
        raw_row = rows_by_key.get((dataset, "passive_raw_lira"))
        if baseline and raw_row and baseline["validation_status"] == "executed" and raw_row["validation_status"] == "executed":
            improved = (raw_row["epsilon_lower_conservative"] or 0.0) >= (baseline["epsilon_lower_conservative"] or 0.0)
            add_check(checks, check_id=f"raw_lira_not_weaker::{dataset}", status="pass" if improved else "warn", description="Raw LiRA is at least as strong as the passive baseline on the canonical image datasets.", scope=dataset, details=f"baseline={baseline['epsilon_lower_conservative']}, raw_lira={raw_row['epsilon_lower_conservative']}")
    adult_raw = rows_by_key.get(("adult", "passive_raw_lira"))
    if adult_raw:
        adult_upper = adult_raw.get("epsilon_upper_tighter")
        adult_lower = adult_raw.get("epsilon_lower_conservative")
        adult_overshoot = (
            adult_upper is not None
            and adult_lower is not None
            and adult_lower > adult_upper + 0.05
        )
        if adult_overshoot:
            add_check(checks, check_id="adult_raw_lira_flagged", status="pass" if adult_raw["result_trust"] == "invalidated" and "pathological_distribution" in adult_raw["diagnostic_tags"] else "fail", description="Adult Raw LiRA overshoot is surfaced explicitly when it occurs.", scope="adult::passive_raw_lira", details=f"trust={adult_raw['result_trust']}, tags={'|'.join(adult_raw['diagnostic_tags'])}")
        else:
            add_check(checks, check_id="adult_raw_lira_flagged", status="pass" if adult_raw["validation_status"] == "executed" else "fail", description="Adult Raw LiRA remains a normal executed row when no overshoot is detected.", scope="adult::passive_raw_lira", details=f"trust={adult_raw['result_trust']}, tags={'|'.join(adult_raw['diagnostic_tags'])}")
    return checks


def build_report(training_rows: list[dict[str, Any]], audit_rows: list[dict[str, Any]], checks: list[dict[str, Any]]) -> str:
    check_counts = Counter(check["status"] for check in checks)
    trust_counts = Counter(row["result_trust"] for row in audit_rows)
    executed_rows = [row for row in audit_rows if row["validation_status"] == "executed"]
    best_rows = sorted([row for row in executed_rows if row["result_trust"] in {"provisional", "exploratory"}], key=lambda row: -float(row.get("epsilon_lower_conservative") or 0.0))
    if check_counts.get("fail", 0) > 0:
        framework_outcome = "fail"
        scientific_status = "invalid for interpretation"
    elif check_counts.get("warn", 0) > 0:
        framework_outcome = "pass with warnings"
        scientific_status = "still provisional overall"
    else:
        framework_outcome = "pass with findings"
        scientific_status = "still provisional overall"
    lines = [
        "# Framework Validation Report", "", "## Outcome", "",
        f"- framework execution validation: `{framework_outcome}`",
        f"- exact-PLD accounting validation: `{'pass' if check_counts.get('fail', 0) == 0 else 'conditional'}`",
        f"- attack semantics validation: `{'pass' if check_counts.get('fail', 0) == 0 else 'conditional'}`",
        f"- scientific trust level: `{scientific_status}`", "", "## Counts", "",
        f"- training rows: `{len(training_rows)}`", f"- audit rows: `{len(audit_rows)}`",
        f"- checks passed: `{check_counts.get('pass', 0)}`", f"- checks warned: `{check_counts.get('warn', 0)}`",
        f"- checks failed: `{check_counts.get('fail', 0)}`", f"- provisional rows: `{trust_counts.get('provisional', 0)}`",
        f"- exploratory rows: `{trust_counts.get('exploratory', 0)}`", f"- invalidated rows: `{trust_counts.get('invalidated', 0)}`",
        f"- not applicable rows: `{trust_counts.get('not_applicable', 0)}`", "", "## Main Findings", "",
    ]
    for row in best_rows[:5]:
        lines.append("- " + f"`{row['dataset']} + {row['attack_name']}` ({support_profile(row)}): " + f"`eps_lower_cons={row.get('epsilon_lower_conservative'):.6f}`, " + f"`eps_upper={row.get('epsilon_upper_tighter'):.6f}`, " + f"`trust={row['result_trust']}`")
    return "\n".join(lines)


def bundle_artifacts(paths: list[Path], zip_path: Path) -> Path:
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as handle:
        for path in paths:
            if path.exists():
                handle.write(path, arcname=path.name)
    return zip_path


PASSIVE_AUDIT_SEEDS = list(range(401, 411))
CANARY_AUDIT_SEEDS = list(range(101, 111))
QUERY_BUDGET = 2048
RAW_LIRA_K = 32
SHADOW_SUBSET_FRACTION = 0.75
NUM_CANARIES = 128

# Fast confirmation profile: MNIST plus built-in sklearn Digits by default.
# CIFAR-10 and Adult remain available, but they depend on external dataset hosts.
RUN_SKLEARN_DIGITS_OPTIONAL = True
RUN_CIFAR10_OPTIONAL = False
RUN_ADULT_OPTIONAL = False


## Step 10: Download Datasets First

This warms the dataset caches and prepares the Adult `.npz` file before the full experiment run.


In [ ]:
def safe_prepare_dataset(label: str, prepare_fn):
    try:
        value = prepare_fn()
        print(f'{label} cache ready: {value}')
        return value
    except Exception as exc:
        print(f'WARNING: {label} cache preparation failed: {exc}')
        print('The full validation cell will continue with any datasets that are available.')
        return None


mnist_info = safe_prepare_dataset('MNIST', lambda: ensure_mnist_npz(DATA_DIR / 'mnist_train.npz'))
if RUN_SKLEARN_DIGITS_OPTIONAL:
    digits_info = safe_prepare_dataset('sklearn_digits', lambda: ensure_sklearn_digits_npz(DATA_DIR / 'sklearn_digits.npz'))
if RUN_CIFAR10_OPTIONAL:
    cifar_info = safe_prepare_dataset('CIFAR-10', lambda: ensure_cifar10_npz(DATA_DIR / 'cifar10_train.npz'))
if RUN_ADULT_OPTIONAL:
    adult_info = safe_prepare_dataset('Adult', lambda: ensure_adult_npz(DATA_DIR / 'adult.npz'))


## Step 11: Run The Full Validation Matrix

This cell trains the canonical GPU-scale configurations, computes the passive and canary audits, validates the results, and writes all output artifacts.


In [ ]:
def make_scaled_spec(*, dataset_name: str, model_name: str, input_dim: int, hidden_dim: int, num_classes: int, learning_rate: float, momentum: float, batch_size: int, eval_batch_size: int, epochs: int) -> dict[str, Any]:
    return {
        "experiment_name": f"colab_gpu_scale_{dataset_name}",
        "dataset_name": dataset_name,
        "model_name": model_name,
        "input_dim": input_dim,
        "hidden_dim": hidden_dim,
        "num_classes": num_classes,
        "learning_rate": learning_rate,
        "momentum": momentum,
        "batch_size": batch_size,
        "eval_batch_size": eval_batch_size,
        "epochs": epochs,
        "clipping_norm": 1.0,
        "noise_multiplier": 1.1,
        "delta": 1e-5,
        "split_seed": 11,
        "training_seed": 123,
    }


def make_attack_error_row(dataset_name: str, attack_name: str, exc: Exception) -> dict[str, Any]:
    return {
        "dataset": dataset_name,
        "attack": attack_name,
        "support_label": "gpu_scale",
        "status": "error",
        "error": str(exc),
        "traceback": traceback.format_exc(),
    }


dataset_specs = [
    {"name": "mnist", "spec": make_scaled_spec(dataset_name="mnist", model_name="simple_mlp", input_dim=784, hidden_dim=256, num_classes=10, learning_rate=0.15, momentum=0.0, batch_size=256, eval_batch_size=512, epochs=5), "train_limit": 60000, "eval_limit": 10000},
]
if RUN_SKLEARN_DIGITS_OPTIONAL:
    dataset_specs.append({"name": "sklearn_digits", "spec": make_scaled_spec(dataset_name="sklearn_digits", model_name="simple_mlp", input_dim=64, hidden_dim=64, num_classes=10, learning_rate=0.2, momentum=0.0, batch_size=128, eval_batch_size=256, epochs=12), "train_limit": 1600, "eval_limit": 400, "validation_fraction": 0.2})
if RUN_CIFAR10_OPTIONAL:
    dataset_specs.append({"name": "cifar10", "spec": make_scaled_spec(dataset_name="cifar10", model_name="cnn_cifar10", input_dim=3072, hidden_dim=256, num_classes=10, learning_rate=0.05, momentum=0.9, batch_size=256, eval_batch_size=512, epochs=8), "train_limit": 50000, "eval_limit": 10000})
if RUN_ADULT_OPTIONAL:
    try:
        adult_input_dim, adult_num_classes, adult_rows = ensure_adult_npz(DATA_DIR / "adult.npz")
        dataset_specs.append({"name": "adult", "spec": make_scaled_spec(dataset_name="adult", model_name="tabular_mlp", input_dim=adult_input_dim, hidden_dim=128, num_classes=adult_num_classes, learning_rate=0.05, momentum=0.0, batch_size=512, eval_batch_size=1024, epochs=5), "train_limit": 30000, "eval_limit": 10000})
    except Exception as exc:
        print(f'WARNING: skipping Adult dataset because preparation failed: {exc}')


started = time.time()
training_rows: list[dict[str, Any]] = []
raw_rows: list[dict[str, Any]] = []

for item in dataset_specs:
    dataset_name = item["name"]
    spec = item["spec"]
    print(f"\n===== {dataset_name} =====")
    try:
        bundle = load_dataset_bundle(dataset_name, DATA_DIR, validation_fraction=item.get("validation_fraction", 0.1), split_seed=spec["split_seed"])
        bundle = subset_bundle(bundle, item["train_limit"], item["eval_limit"], seed=777)
        training_outcome = train_dp_sgd(spec, bundle, save_checkpoint=True, run_descriptor=f"{dataset_name}_gpu_scale", return_model_state=True)
    except Exception as exc:
        training_rows.append({"dataset": dataset_name, "status": "error", "error": str(exc), "traceback": traceback.format_exc()})
        for attack_name in ["passive_negative_loss", "passive_negative_loss_matched", "passive_raw_lira", "canary_random"]:
            raw_rows.append(make_attack_error_row(dataset_name, attack_name, exc))
        continue

    record = training_outcome.record
    training_rows.append({
        "dataset": dataset_name,
        "status": "ok",
        "train_size": bundle.train_size,
        "eval_size": bundle.eval_size,
        "epsilon_upper_rdp": record.epsilon_upper_theory,
        "epsilon_upper_tighter": record.epsilon_upper_pld,
        "upper_bound_backend": record.pld_accounting_backend,
        "accuracy": record.utility_metrics.get("accuracy"),
        "loss": record.utility_metrics.get("loss"),
        "training_run_id": record.training_run_id,
    })

    passive_ready = False
    device = torch.device("cuda")
    try:
        target_model = load_model_from_state_dict(spec, training_outcome.model_state_dict, device=device)
        query_train_indices, query_eval_indices, per_seed_train, per_seed_eval = sample_query_indices(
            train_size=len(bundle.train_dataset),
            eval_size=len(bundle.eval_dataset),
            audit_seeds=PASSIVE_AUDIT_SEEDS,
            query_budget=QUERY_BUDGET,
        )
        target_train_losses = compute_loss_for_indices(target_model, bundle.train_dataset, query_train_indices, device=device, batch_size=spec["eval_batch_size"])
        target_eval_losses = compute_loss_for_indices(target_model, bundle.eval_dataset, query_eval_indices, device=device, batch_size=spec["eval_batch_size"])
        passive_ready = True
    except Exception as exc:
        raw_rows.append(make_attack_error_row(dataset_name, "passive_negative_loss", exc))
        raw_rows.append(make_attack_error_row(dataset_name, "passive_negative_loss_matched", exc))
        raw_rows.append(make_attack_error_row(dataset_name, "passive_raw_lira", exc))

    if passive_ready:
        try:
            nl_member_scores, nl_nonmember_scores = negative_loss_scores(
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                per_seed_train=per_seed_train,
                per_seed_eval=per_seed_eval,
                target_train_losses=target_train_losses,
                target_eval_losses=target_eval_losses,
                audit_seeds=PASSIVE_AUDIT_SEEDS,
            )
            nl_estimate = estimate_empirical_lower_bound(
                member_scores=nl_member_scores,
                nonmember_scores=nl_nonmember_scores,
                delta=spec["delta"],
                score_direction="higher_is_member",
                align_event_to_score_direction=True,
                require_member_favoring=True,
                report_confidence_supported_lower_bound=True,
            )
            passive_row = {
                "dataset": dataset_name,
                "attack": "passive_negative_loss",
                "support_label": "gpu_scale",
                "status": "ok",
                "audit_regime": "passive_final_model_only",
                "score_name": "negative_loss",
                "query_budget_per_seed": QUERY_BUDGET,
                "num_audit_seeds": len(PASSIVE_AUDIT_SEEDS),
                "audit_seeds": json.dumps(PASSIVE_AUDIT_SEEDS),
                "epsilon_upper_rdp": record.epsilon_upper_theory,
                "epsilon_upper_tighter": record.epsilon_upper_pld,
                "tighter_upper_backend": record.pld_accounting_backend,
                "epsilon_lower_conservative": nl_estimate.epsilon_lower_empirical,
                "epsilon_lower_point": nl_estimate.epsilon_lower_empirical_point_estimate,
                "selected_tpr": nl_estimate.selected_tpr,
                "selected_fpr": nl_estimate.selected_fpr,
                "warning": nl_estimate.warning_message,
                "num_member_samples": nl_estimate.num_member_samples,
                "num_nonmember_samples": nl_estimate.num_nonmember_samples,
            }
            raw_rows.append(passive_row)
            raw_rows.append(dict(passive_row, attack="passive_negative_loss_matched"))
        except Exception as exc:
            raw_rows.append(make_attack_error_row(dataset_name, "passive_negative_loss", exc))
            raw_rows.append(make_attack_error_row(dataset_name, "passive_negative_loss_matched", exc))

        try:
            shadow_train_losses, shadow_eval_losses, shadow_member_sets = train_shadow_losses(
                spec=spec,
                bundle=bundle,
                dataset_name=dataset_name,
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                k_max=RAW_LIRA_K,
                shadow_subset_fraction=SHADOW_SUBSET_FRACTION,
            )
            raw_member_scores, raw_nonmember_scores = raw_lira_scores(
                query_train_indices=query_train_indices,
                query_eval_indices=query_eval_indices,
                per_seed_train=per_seed_train,
                per_seed_eval=per_seed_eval,
                target_train_losses=target_train_losses,
                target_eval_losses=target_eval_losses,
                shadow_train_losses=shadow_train_losses,
                shadow_eval_losses=shadow_eval_losses,
                shadow_member_sets=shadow_member_sets,
                k=RAW_LIRA_K,
                audit_seeds=PASSIVE_AUDIT_SEEDS,
            )
            raw_estimate = estimate_empirical_lower_bound(
                member_scores=raw_member_scores,
                nonmember_scores=raw_nonmember_scores,
                delta=spec["delta"],
                score_direction="lower_is_member",
                align_event_to_score_direction=True,
                require_member_favoring=True,
                report_confidence_supported_lower_bound=True,
            )
            raw_rows.append({
                "dataset": dataset_name,
                "attack": "passive_raw_lira",
                "support_label": "gpu_scale",
                "status": "ok",
                "k_shadows": RAW_LIRA_K,
                "query_budget_per_seed": QUERY_BUDGET,
                "num_audit_seeds": len(PASSIVE_AUDIT_SEEDS),
                "audit_seeds": json.dumps(PASSIVE_AUDIT_SEEDS),
                "score_name": "raw_lira_score",
                "epsilon_upper_rdp": record.epsilon_upper_theory,
                "epsilon_upper_tighter": record.epsilon_upper_pld,
                "tighter_upper_backend": record.pld_accounting_backend,
                "epsilon_lower_conservative": raw_estimate.epsilon_lower_empirical,
                "epsilon_lower_point": raw_estimate.epsilon_lower_empirical_point_estimate,
                "selected_tpr": raw_estimate.selected_tpr,
                "selected_fpr": raw_estimate.selected_fpr,
                "warning": raw_estimate.warning_message,
                "num_member_samples": raw_estimate.num_member_samples,
                "num_nonmember_samples": raw_estimate.num_nonmember_samples,
            })
        except Exception as exc:
            raw_rows.append(make_attack_error_row(dataset_name, "passive_raw_lira", exc))

    try:
        raw_rows.append(
            run_canary_random_audit(
                spec=spec,
                base_bundle=bundle,
                target_record=record,
                num_canaries=NUM_CANARIES,
                audit_seeds=CANARY_AUDIT_SEEDS,
            )
        )
    except Exception as exc:
        raw_rows.append(make_attack_error_row(dataset_name, "canary_random", exc))

normalized_rows = [normalize_attack_row(row) for row in raw_rows]
checks = build_checks(training_rows, normalized_rows)
report_text = build_report(training_rows, normalized_rows, checks)
summary_data = {
    "scope": "colab gpu-scale framework validation",
    "cuda_device": torch.cuda.get_device_name(0),
    "training_rows": training_rows,
    "audit_rows": normalized_rows,
    "checks": checks,
    "elapsed_seconds": round(time.time() - started, 3),
}
save_json(SUMMARY_JSON, summary_data)
write_csv(SUMMARY_CSV, normalized_rows)
save_json(CHECKS_JSON, checks)
REPORT_MD.write_text(report_text, encoding="utf-8")
bundle_artifacts([SUMMARY_JSON, SUMMARY_CSV, CHECKS_JSON, REPORT_MD], ARTIFACTS_ZIP)
print("summary_json =", SUMMARY_JSON)
print("rows_csv =", SUMMARY_CSV)
print("checks_json =", CHECKS_JSON)
print("report_md =", REPORT_MD)
print("artifacts_zip =", ARTIFACTS_ZIP)
print("elapsed_seconds =", summary_data["elapsed_seconds"])


## Step 12: Inspect Results In-Notebook

This reads the written artifacts back into the notebook, shows the main tables, and renders the Markdown report.


In [ ]:
from IPython.display import Markdown, display

if not SUMMARY_JSON.exists():
    raise FileNotFoundError(SUMMARY_JSON)

summary_data = json.loads(SUMMARY_JSON.read_text(encoding='utf-8'))
checks = summary_data.get('checks', [])
training_rows = summary_data.get('training_rows', [])
audit_rows = summary_data.get('audit_rows', [])

print('Framework validation outcome summary')
print('- cuda_device:', summary_data.get('cuda_device'))
print('- elapsed_seconds:', summary_data.get('elapsed_seconds'))
print('- training_rows:', len(training_rows))
print('- audit_rows:', len(audit_rows))
print('- checks_passed:', sum(1 for c in checks if c.get('status') == 'pass'))
print('- checks_warned:', sum(1 for c in checks if c.get('status') == 'warn'))
print('- checks_failed:', sum(1 for c in checks if c.get('status') == 'fail'))
print('- artifacts_zip:', ARTIFACTS_ZIP)

training_df = pd.DataFrame(training_rows)
audit_df = pd.DataFrame(audit_rows)
checks_df = pd.DataFrame(checks)

if not training_df.empty:
    display(training_df)

if not audit_df.empty:
    columns = [
        'dataset',
        'attack_name',
        'epsilon_lower_conservative',
        'epsilon_upper_tighter',
        'result_trust',
        'validation_status',
        'diagnostic_tags',
    ]
    present = [column for column in columns if column in audit_df.columns]
    display(audit_df[present])

failed_checks = checks_df[checks_df['status'] == 'fail'] if not checks_df.empty and 'status' in checks_df.columns else checks_df.iloc[0:0]
if not failed_checks.empty:
    print('\nFailed checks')
    display(failed_checks)

if REPORT_MD.exists():
    display(Markdown(REPORT_MD.read_text(encoding='utf-8')))
